# Lesson 1.3 — Selecting the Architectural Pattern

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish augmented LLM, workflow, and agentic patterns and their cost/risk profiles
- Apply a selection rubric that maps problem traits to a pattern
- Justify why the simplest sufficient pattern is usually correct

**You will produce:** A reusable `select_pattern()` rubric and a decision table over several scenarios.

## Concepts

Anthropic's guidance: **use the simplest pattern that works**; add autonomy only when the problem demands it. Three tiers:

- **Augmented LLM** — a single call with retrieval and/or tools. Predictable, cheap, easy to evaluate. Default for well-scoped tasks.
- **Workflow** — multiple LLM calls on *predefined* code paths (prompt chaining, routing, parallelization). You control the control flow; the model fills steps.
- **Agentic** — the *model* decides the control flow and which tools to call in a loop. Maximum flexibility, maximum cost/latency/variance and the hardest to evaluate and secure.

Choosing agentic when a workflow suffices is the most common senior-level over-engineering error. The exam tests whether you can match pattern to problem traits: task complexity, need for dynamic decisions, tolerance for latency/cost, and auditability requirements.

> **Reference:** [Anthropic — Building effective agents](https://docs.anthropic.com/en/docs/build-with-claude/building-effective-agents)

In [ ]:
# !pip install anthropic
# Guarded Claude client setup — every lesson runs with or without an API key.
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic()
        LIVE = True
    else:
        client = None
        LIVE = False
except Exception as e:  # SDK not installed in this kernel
    client = None
    LIVE = False
    print("anthropic SDK unavailable:", e)

# Current Claude model family (confirm exact IDs at https://docs.anthropic.com/en/docs/about-claude/models)
MODELS = {
    "opus": "claude-opus-4-20250514",    # deepest reasoning, highest cost/latency
    "sonnet": "claude-sonnet-4-20250514",# balanced default workhorse
    "haiku": "claude-3-5-haiku-20241022",# fastest/cheapest for simple tasks
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    """Live call when a key exists; otherwise a deterministic mock so notebooks run."""
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system or "You are a helpful solution architect.",
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Encode the selection rubric

We score each problem on four traits and map the profile to a pattern. The rubric is deliberately conservative — it only recommends agentic when dynamic decision-making is genuinely required.

In [ ]:
def select_pattern(dynamic_control_flow, num_subtasks, auditability_critical, latency_budget_s):
    """Return (pattern, rationale) from problem traits."""
    if dynamic_control_flow and latency_budget_s >= 10 and not auditability_critical:
        return "agentic", "Model must choose steps/tools dynamically; latency allows a loop"
    if num_subtasks >= 2:
        return "workflow", "Multiple predefined steps; keep control flow in code for auditability"
    return "augmented_llm", "Single well-scoped task; one call with retrieval/tools is enough"

print(select_pattern(True, 4, False, 30))
print(select_pattern(False, 3, True, 20))
print(select_pattern(False, 1, True, 5))

## Code-Along: Step 2 — Build a scenario decision table

Run the rubric over realistic scenarios to see the pattern boundaries. A table like this is exactly how you communicate a recommendation to stakeholders.

In [ ]:
scenarios = [
  ("Classify support ticket priority", False, 1, True, 3),
  ("Extract+validate+summarize a contract", False, 3, True, 30),
  ("Open-ended research assistant", True, 5, False, 60),
  ("Route then answer FAQ", False, 2, False, 5),
]
print(f"{'scenario':38} {'pattern':14} rationale")
for name, dyn, n, audit, lat in scenarios:
    pat, why = select_pattern(dyn, n, audit, lat)
    print(f"{name:38} {pat:14} {why}")

## Exercise

The rubric currently ignores **cost sensitivity**. Add a `cost_sensitive` flag so that a cost-sensitive problem never selects `agentic` (it should fall back to `workflow`). Add two new scenarios and show the table changes.

In [ ]:
# TODO: add cost_sensitive parameter to select_pattern and adjust the logic
def select_pattern_v2(dynamic_control_flow, num_subtasks, auditability_critical, latency_budget_s, cost_sensitive):
    pass  # TODO

# TODO: rerun the decision table with two added cost-sensitive scenarios

## Key Takeaways

- Default to the simplest pattern; escalate to agentic only for dynamic control flow.
- Workflows keep control flow in code — better for auditability and evaluation.
- Agentic systems trade predictability for flexibility; price that in.
- A rubric turns pattern choice into a defensible, repeatable decision.

---

**Next:** [Lesson 1.4 — Multi-Agent Systems and Orchestration](./lesson-1.4-multi-agent-orchestration.ipynb)